# sPCA component scores across barcode-calling cutoffs

Re-runs the production `waggr` scoring (`analysis/spca/spca_waggr_scoring.ipynb`) on the doublet
population called at each of DSB4-9, against the *existing* `0.1_alpha1.0` loadings - the question
is about scores, not about the components.

In [ ]:
# Repo-relative paths: inputs are read from imports_stable/, outputs are written to analysis_outs/
import os
from pathlib import Path
_here = Path.cwd().resolve()
REPO_DIR = str(next(p for p in [_here, *_here.parents] if (p / "imports_stable").is_dir()))
IMPORTS_DIR = f"{REPO_DIR}/imports_stable"
OUTS_DIR = f"{REPO_DIR}/analysis_outs/02_combinatorial_screen_signalseq_SIG13"
for _d in ["qc_barcode_cutoff"]:
    os.makedirs(f"{OUTS_DIR}/{_d}", exist_ok=True)

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import decoupler as dc

# not in imports_stable (size); generate it with 03_generate_cutoff_datasets.ipynb first
H5AD = OUTS_DIR + "/scanpy_outs/cutoff_sweep/SIG13_cutoffSweep_spcaGenes_log1pnorm.h5ad"
SPCA_LOADINGS = IMPORTS_DIR + "/SIG13/analysis_outs/spca/zscore_degs_allLigands_0.1_alpha1.0_sPCA_loadings.csv"
PROD_SCORES = IMPORTS_DIR + "/SIG13/analysis_outs/spca/degs_zscore_allLigands/zscore_degs_allLigands_0.1_alpha1.0_waggr_score.csv"
OUT_DIR = OUTS_DIR + "/qc_barcode_cutoff"

CUTOFFS = list(range(1, 11))
REF_CUTOFF = 7

## Import

In [2]:
rna = sc.read_h5ad(H5AD)
rna

AnnData object with n_obs × n_vars = 503574 × 6610
    obs: 'lane', 'replicate', 'total_counts', 'n_genes_by_counts', 'pct_counts_mt', 'keep_DSB1', 'ligand_call_DSB1', 'num_umis_DSB1', 'keep_DSB2', 'ligand_call_DSB2', 'num_umis_DSB2', 'keep_DSB3', 'ligand_call_DSB3', 'num_umis_DSB3', 'keep_DSB4', 'ligand_call_DSB4', 'num_umis_DSB4', 'keep_DSB5', 'ligand_call_DSB5', 'num_umis_DSB5', 'keep_DSB6', 'ligand_call_DSB6', 'num_umis_DSB6', 'keep_DSB7', 'ligand_call_DSB7', 'num_umis_DSB7', 'keep_DSB8', 'ligand_call_DSB8', 'num_umis_DSB8', 'keep_DSB9', 'ligand_call_DSB9', 'num_umis_DSB9', 'keep_DSB10', 'ligand_call_DSB10', 'num_umis_DSB10'
    var: 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'

In [3]:
spca_components = pd.read_csv(SPCA_LOADINGS)
net = spca_components.rename(columns={'gene': 'target',
                                      'spca_component': 'source',
                                      'loading': 'weight'})
net = net[net['target'].isin(rna.var_names)]
print(f"{net['target'].nunique()} genes across {net['source'].nunique()} components")

6610 genes across 78 components


## Score every cutoff

Each cutoff is scaled within its own cell population, matching the production scoring, which
scaled on the DSB7 population alone.

`times=0` (no permutation p-values) and `tmin=5` match the production `waggr` call. Pseudobulk
unit is `(ligand_call, replicate)` at that cutoff.

In [4]:
score_frames, qc_frames = [], []

for k in CUTOFFS:
    call_col = f"ligand_call_DSB{k}"
    mask = rna.obs[f"keep_DSB{k}"].values.astype(bool)
    sub = rna[mask].copy()
    sc.pp.scale(sub)

    pb = sc.get.aggregate(sub, by=[call_col, 'replicate'], func='mean')
    dc.mt.waggr(pb, net, tmin=5, layer='mean', times=0, verbose=False)

    sdf = pd.DataFrame(np.asarray(pb.obsm['score_waggr']),
                       index=pb.obs_names,
                       columns=pb.obsm['score_waggr'].columns)
    long = (sdf.rename_axis('unit_id')
               .reset_index()
               .melt(id_vars='unit_id', var_name='component', value_name='score'))
    long = long.join(pb.obs[[call_col, 'replicate']].astype(str).rename(
        columns={call_col: 'ligand_call'}), on='unit_id')
    long['cutoff'] = k
    score_frames.append(long)

    qc = (sub.obs.groupby([call_col, 'replicate'], observed=True)
          .agg(n_cells=('total_counts', 'size'),
               mean_total_counts=('total_counts', 'mean'),
               mean_n_genes=('n_genes_by_counts', 'mean'),
               mean_pct_mt=('pct_counts_mt', 'mean'))
          .reset_index()
          .rename(columns={call_col: 'ligand_call'}))
    qc['cutoff'] = k
    qc['unit_id'] = qc['ligand_call'].astype(str) + '_' + qc['replicate'].astype(str)
    qc_frames.append(qc)

    print(f"DSB{k}: {sub.n_obs} cells, {pb.n_obs} units x {sdf.shape[1]} components")
    del sub

scores_long = pd.concat(score_frames, ignore_index=True)
subset_qc = pd.concat(qc_frames, ignore_index=True)
scores_long.shape

/data1/rudenska/EYW/mamba_envs/scanpy_standard2/lib/python3.14/functools.py:982: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


DSB1: 81057 cells, 1314 units x 78 components


/data1/rudenska/EYW/mamba_envs/scanpy_standard2/lib/python3.14/functools.py:982: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


DSB2: 160182 cells, 1320 units x 78 components


/data1/rudenska/EYW/mamba_envs/scanpy_standard2/lib/python3.14/functools.py:982: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


...

/data1/rudenska/EYW/mamba_envs/scanpy_standard2/lib/python3.14/functools.py:982: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


DSB9: 236273 cells, 1318 units x 78 components


/data1/rudenska/EYW/mamba_envs/scanpy_standard2/lib/python3.14/functools.py:982: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


DSB10: 169075 cells, 1309 units x 78 components


(1028118, 6)

## Keep units present at every cutoff

Units are paired across cutoffs so the comparisons in notebook 7 are like-for-like.

In [5]:
units_per_cutoff = scores_long.groupby('cutoff')['unit_id'].unique()
shared_units = set.intersection(*[set(u) for u in units_per_cutoff])
dropped = sorted(set(scores_long['unit_id']) - shared_units)
print(f"units kept: {len(shared_units)}  dropped (absent at >= 1 cutoff): {len(dropped)}")
print(dropped)

scores_long = scores_long[scores_long['unit_id'].isin(shared_units)]
subset_qc = subset_qc[subset_qc['unit_id'].isin(shared_units)]

units kept: 1303  dropped (absent at >= 1 cutoff): 17
['IL23_CCL20_rep1', 'IL23_CCL21A_rep1', 'IL23_IFNB1_rep1', 'IL23_IL1A_rep1', 'IL23_IL1A_rep2', 'IL23_IL1B_rep1', 'IL23_IL33_rep1', 'IL23_IL9_rep1', 'IL23_IL9_rep2', 'IL23_OSM_rep1', 'IL23_TNFSF4_rep1', 'IL23_TNFSF9_rep1', 'IL27_CCL21A_rep1', 'IL27_IFNB1_rep1', 'IL27_IFNK_rep1', 'IL27_IL23_rep1', 'IL6_IFNB1_rep1']


## Pipeline check against the production scores

The DSB7 scores should reproduce the production table: same cells, same loadings, and now the
same scaling population. Anything below a near-perfect correlation means this notebook has
diverged from the production scoring.

In [6]:
prod = pd.read_csv(PROD_SCORES, index_col=0)
prod_long = (prod.rename_axis('unit_id')
                 .reset_index()
                 .melt(id_vars='unit_id', var_name='component', value_name='score_prod'))

chk = (scores_long.query("cutoff == @REF_CUTOFF")
       .merge(prod_long, on=['unit_id', 'component'], how='inner'))
print(f"shared unit x component pairs: {len(chk)}")
print(f"Pearson(DSB7, production) = {chk['score'].corr(chk['score_prod']):.6f}")
print(f"max |difference| = {(chk['score'] - chk['score_prod']).abs().max():.3e}")

shared unit x component pairs: 101634
Pearson(DSB7, production) = 1.000000
max |difference| = 4.441e-16


## Write outputs

In [7]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

scores_long.to_csv(f"{OUT_DIR}/spca_waggr_cutoff_scores_long.csv.gz", index=False)
subset_qc.to_csv(f"{OUT_DIR}/spca_waggr_cutoff_qc.csv", index=False)

print(f"wrote {OUT_DIR}/spca_waggr_cutoff_scores_long.csv.gz")
print(f"wrote {OUT_DIR}/spca_waggr_cutoff_qc.csv")

wrote /data1/rudenska/EYW/git_projects/SIG13/analysis_outs/qc_barcode_cutoff/spca_waggr_cutoff_scores_long.csv.gz
wrote /data1/rudenska/EYW/git_projects/SIG13/analysis_outs/qc_barcode_cutoff/spca_waggr_cutoff_qc.csv
